# Hack-A-Roo 2026, AI for Quantum track
# Notebook 7: Compilation cost predictor

**Problem.** A circuit is written as if any two qubits can interact. Real hardware only lets neighbors interact. The compiler has to pick which physical qubit holds which logical qubit (layout) and insert SWAP gates whenever two logical qubits that need to interact are not neighbors (routing). Every SWAP is three CX gates of extra noise. Compiling is expensive and the result depends on heuristics. Can a model predict the SWAP cost of a circuit from its structure before compiling it, and can that prediction guide a better layout?

**What this notebook already does.** It builds a square-lattice coupling map, generates random circuits with random two-qubit interaction graphs, compiles each one with qiskit's transpiler onto the lattice, and records the SWAP count, the output CX count, the depth, and the compile time. It computes cheap features of each circuit's interaction graph and trains a regressor from those features to the SWAP count, scored on unseen circuits. Two baselines isolate what the graph features add: a linear model on CX count alone, and the same gradient-boosting model on CX count alone, so the comparison that matters holds the model family fixed and changes only the inputs.

**Metric.** Mean absolute error in predicted SWAP count on held-out circuits, all-features boosting against CX-only boosting.

**Hypothesis to test.** Interaction-graph structure predicts routing cost beyond what the gate count alone predicts, for this compiler configuration, this lattice, and this circuit distribution. The notebook does not assume this, and the result does not transfer to other compilers, hardware shapes, or circuit families without rerunning.

**Deliverable for the poster.** One figure (predicted versus actual SWAP count on held-out circuits, CX-only boosting against all-features boosting) and the error numbers. Then whichever shred-out you pick at the bottom.

Run every cell top to bottom once. The base run takes under a minute. Then go to the Shred-outs section.

In [ ]:
# CONFIG CELL. Every constant, seed, and path lives here. Nothing is hardcoded downstream.

RANDOM_SEED = 42

# Hardware: a square lattice of physical qubits with nearest-neighbor coupling.
LATTICE_ROWS = 4
LATTICE_COLUMNS = 4

# Random circuits. Each circuit draws a logical qubit count and a CX count from these ranges, then picks each
# CX's pair of qubits uniformly at random. Single-qubit rotations are interleaved so the circuit is not pure CX.
NUMBER_OF_CIRCUITS = 400
LOGICAL_QUBIT_COUNT_LOW = 4
LOGICAL_QUBIT_COUNT_HIGH = 16          # cannot exceed LATTICE_ROWS * LATTICE_COLUMNS
CX_COUNT_LOW = 5
CX_COUNT_HIGH = 80

# Compiler
OPTIMIZATION_LEVEL = 1
BASIS_GATES = ["rz", "sx", "x", "cx", "swap"]    # swap kept in the basis so routing SWAPs are countable by name. Hardware has no swap gate: each one is 3 CX.
TRANSPILER_SEED = 0                               # SABRE routing is randomized; this pins it. One seed = one draw of the compiler's dice.
CX_PER_SWAP = 3

# Learning
TEST_FRACTION = 0.25
GRADIENT_BOOSTING_TREES = 300
GRADIENT_BOOSTING_DEPTH = 3

# Output
OUTPUT_PATH = "compilation_cost_dataset.pkl"
FIGURE_PATH = "compilation_cost_figure.png"

In [ ]:
# Install. Skip this cell if your environment already has these. Tested with qiskit 2.6, scikit-learn 1.8 and 1.9, networkx 3.7.
!pip install -q "qiskit[visualization]" "scikit-learn>=1.5" "networkx>=3.2" matplotlib

In [ ]:
import time
import pickle
import numpy as np
import sklearn
import qiskit
import networkx as nx
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import CouplingMap
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

random_generator = np.random.default_rng(RANDOM_SEED)
PHYSICAL_QUBIT_COUNT = LATTICE_ROWS * LATTICE_COLUMNS
assert LOGICAL_QUBIT_COUNT_HIGH <= PHYSICAL_QUBIT_COUNT
package_versions = {"numpy": np.__version__, "qiskit": qiskit.__version__, "scikit-learn": sklearn.__version__, "networkx": nx.__version__}
print("imports done.", PHYSICAL_QUBIT_COUNT, "physical qubits", package_versions)

In [ ]:
# The coupling map, built explicitly as an edge list on the square lattice.
# Physical qubit at (row, column) has index row * LATTICE_COLUMNS + column.
# Edges are listed in both directions because qiskit's CouplingMap is directed and CX can run either way here.

def lattice_index(row, column):
    """Physical qubit index of a lattice position"""
    return row * LATTICE_COLUMNS + column


coupling_edges = []
for row in range(LATTICE_ROWS):
    for column in range(LATTICE_COLUMNS):
        if column + 1 < LATTICE_COLUMNS:
            coupling_edges.append([lattice_index(row, column), lattice_index(row, column + 1)])
            coupling_edges.append([lattice_index(row, column + 1), lattice_index(row, column)])
        if row + 1 < LATTICE_ROWS:
            coupling_edges.append([lattice_index(row, column), lattice_index(row + 1, column)])
            coupling_edges.append([lattice_index(row + 1, column), lattice_index(row, column)])

coupling_map = CouplingMap(couplinglist=coupling_edges)
assert coupling_map.size() == PHYSICAL_QUBIT_COUNT
print(f"{LATTICE_ROWS}x{LATTICE_COLUMNS} lattice, {len(coupling_edges) // 2} undirected couplings")
print("neighbors of physical qubit 5:", sorted(coupling_map.neighbors(5)))

In [ ]:
# Random circuits with random interaction graphs, built gate by gate.
# The interaction graph has one node per logical qubit and one edge per distinct pair that shares a CX.
# That graph, not the gate list, is what the features describe.

def build_random_circuit(logical_qubit_count, cx_count):
    """Random CX pairs with a random single-qubit rotation before each CX on each of its qubits"""
    circuit = QuantumCircuit(logical_qubit_count)
    for cx_index in range(cx_count):
        control, target = random_generator.choice(logical_qubit_count, size=2, replace=False)
        circuit.rz(random_generator.uniform(0, 2 * np.pi), int(control))
        circuit.sx(int(target))
        circuit.cx(int(control), int(target))
    return circuit


def interaction_graph_of(circuit):
    """Undirected graph of logical qubits with an edge for each pair that shares at least one CX"""
    graph = nx.Graph()
    graph.add_nodes_from(range(circuit.num_qubits))
    for instruction in circuit.data:
        if instruction.operation.name == "cx":
            qubit_a = circuit.find_bit(instruction.qubits[0]).index
            qubit_b = circuit.find_bit(instruction.qubits[1]).index
            graph.add_edge(qubit_a, qubit_b)
    return graph


FEATURE_NAMES = [
    "logical_qubit_count",
    "cx_count",
    "distinct_pair_count",
    "interaction_density",
    "max_degree",
    "mean_degree",
    "degree_std",
    "triangle_count",
    "is_planar",
    "input_depth",
]


def circuit_features(circuit):
    """Cheap features of the circuit and its interaction graph, in FEATURE_NAMES order"""
    graph = interaction_graph_of(circuit)
    degrees = np.array([degree for node, degree in graph.degree()])
    planar, embedding = nx.check_planarity(graph)
    return [
        circuit.num_qubits,
        circuit.count_ops().get("cx", 0),
        graph.number_of_edges(),
        nx.density(graph),
        int(degrees.max()),
        float(degrees.mean()),
        float(degrees.std()),
        sum(nx.triangles(graph).values()) / 3,
        int(planar),
        circuit.depth(),
    ]


def compile_to_lattice(circuit):
    """Transpiles onto the lattice and returns (swap count, output depth with swap as one gate, output cx count, seconds)"""
    # The depth returned here counts each SWAP as one instruction. Physical depth after decomposing SWAP into 3 CX
    # is larger and depends on further optimization; even that is gate depth, not wall-clock duration.
    start_time = time.time()
    compiled = transpile(circuit, coupling_map=coupling_map, basis_gates=BASIS_GATES,
                         optimization_level=OPTIMIZATION_LEVEL, seed_transpiler=TRANSPILER_SEED)
    seconds = time.time() - start_time
    operation_counts = compiled.count_ops()
    return operation_counts.get("swap", 0), compiled.depth(), operation_counts.get("cx", 0), seconds


example_circuit = build_random_circuit(6, 10)
print(example_circuit.draw(fold=120))
print("features:", dict(zip(FEATURE_NAMES, circuit_features(example_circuit))))
example_swaps, example_depth, example_cx, example_seconds = compile_to_lattice(example_circuit)
print(f"compiled: {example_swaps} swaps, depth {example_circuit.depth()} -> {example_depth}, in {example_seconds:.2f}s")

In [ ]:
# Generate and compile the dataset. This is the slow cell, with progress printing.

circuits = []
feature_rows = []
swap_counts = []
output_cx_counts = []        # CX gates in the routed circuit before SWAP decomposition
output_depths = []           # depth with each SWAP counted as one instruction
input_depths = []
compile_seconds = []         # local timing on this machine, not a portable label

start_time = time.time()
for circuit_index in range(NUMBER_OF_CIRCUITS):
    logical_qubit_count = int(random_generator.integers(LOGICAL_QUBIT_COUNT_LOW, LOGICAL_QUBIT_COUNT_HIGH + 1))
    cx_count = int(random_generator.integers(CX_COUNT_LOW, CX_COUNT_HIGH + 1))
    circuit = build_random_circuit(logical_qubit_count, cx_count)
    swaps, depth_after, cx_after, seconds = compile_to_lattice(circuit)

    circuits.append(circuit)
    feature_rows.append(circuit_features(circuit))
    swap_counts.append(swaps)
    output_cx_counts.append(cx_after)
    output_depths.append(depth_after)
    input_depths.append(circuit.depth())
    compile_seconds.append(seconds)

    if (circuit_index + 1) % 50 == 0:
        elapsed_seconds = time.time() - start_time
        remaining_seconds = elapsed_seconds / (circuit_index + 1) * (NUMBER_OF_CIRCUITS - circuit_index - 1)
        print(f"compiled {circuit_index + 1}/{NUMBER_OF_CIRCUITS}  elapsed {elapsed_seconds:.0f}s  remaining about {remaining_seconds:.0f}s")

feature_rows = np.array(feature_rows, dtype=float)
swap_counts = np.array(swap_counts)
output_cx_counts = np.array(output_cx_counts)
physical_cx_estimate = output_cx_counts + CX_PER_SWAP * swap_counts     # CX count after decomposing every SWAP, before any further optimization
output_depths = np.array(output_depths)
input_depths = np.array(input_depths)
compile_seconds = np.array(compile_seconds)
assert feature_rows.shape == (NUMBER_OF_CIRCUITS, len(FEATURE_NAMES))

print()
print(f"swap counts range from {swap_counts.min()} to {swap_counts.max()}, mean {swap_counts.mean():.1f}")
print(f"physical CX estimate (cx + {CX_PER_SWAP} per swap) is {np.mean(physical_cx_estimate / feature_rows[:, FEATURE_NAMES.index('cx_count')]):.2f} times the input CX count on average")
print(f"depth grows by a factor of {np.mean(output_depths / input_depths):.2f} on average after routing, with each SWAP counted as one instruction")
print(f"mean compile time {compile_seconds.mean():.3f}s per circuit on this machine")

In [ ]:
# Three predictors on the same split. The comparison that isolates the graph features is the second against the third:
# same model family, different inputs.
#   1. CX count alone, linear regression          (the simplest possible baseline)
#   2. CX count alone, gradient boosting           (same model family as 3, so the gap to 3 is due to the inputs)
#   3. all features, gradient boosting

train_indices, test_indices = train_test_split(
    np.arange(NUMBER_OF_CIRCUITS), test_size=TEST_FRACTION, random_state=RANDOM_SEED)

cx_count_column = FEATURE_NAMES.index("cx_count")
cx_only_model = LinearRegression().fit(feature_rows[train_indices][:, [cx_count_column]], swap_counts[train_indices])
cx_only_prediction = cx_only_model.predict(feature_rows[test_indices][:, [cx_count_column]])

cx_only_boosting_model = GradientBoostingRegressor(
    n_estimators=GRADIENT_BOOSTING_TREES, max_depth=GRADIENT_BOOSTING_DEPTH, random_state=RANDOM_SEED)
cx_only_boosting_model.fit(feature_rows[train_indices][:, [cx_count_column]], swap_counts[train_indices])
cx_only_boosting_prediction = cx_only_boosting_model.predict(feature_rows[test_indices][:, [cx_count_column]])

all_features_model = GradientBoostingRegressor(
    n_estimators=GRADIENT_BOOSTING_TREES, max_depth=GRADIENT_BOOSTING_DEPTH, random_state=RANDOM_SEED)
all_features_model.fit(feature_rows[train_indices], swap_counts[train_indices])
all_features_prediction = all_features_model.predict(feature_rows[test_indices])

cx_only_error = np.abs(cx_only_prediction - swap_counts[test_indices])
cx_only_boosting_error = np.abs(cx_only_boosting_prediction - swap_counts[test_indices])
all_features_error = np.abs(all_features_prediction - swap_counts[test_indices])

print(f"RESULTS ON {len(test_indices)} HELD-OUT CIRCUITS (swap count), one compiler seed, one lattice, one circuit family")
print(f"  mean swap count in test set:              {swap_counts[test_indices].mean():.1f}")
print(f"  CX-count-only linear, mean abs error:     {cx_only_error.mean():.2f}")
print(f"  CX-count-only boosting, mean abs error:   {cx_only_boosting_error.mean():.2f}")
print(f"  all-features boosting, mean abs error:    {all_features_error.mean():.2f}   <- compare with the line above: same model, more inputs")
print(f"  all-features R^2:                         {all_features_model.score(feature_rows[test_indices], swap_counts[test_indices]):.3f}")
print("  feature importances:")
for feature_name, importance in zip(FEATURE_NAMES, all_features_model.feature_importances_):
    print(f"    {feature_name:22s} {importance:.3f}")

config = {
    "RANDOM_SEED": RANDOM_SEED,
    "LATTICE_ROWS": LATTICE_ROWS,
    "LATTICE_COLUMNS": LATTICE_COLUMNS,
    "NUMBER_OF_CIRCUITS": NUMBER_OF_CIRCUITS,
    "LOGICAL_QUBIT_COUNT_LOW": LOGICAL_QUBIT_COUNT_LOW,
    "LOGICAL_QUBIT_COUNT_HIGH": LOGICAL_QUBIT_COUNT_HIGH,
    "CX_COUNT_LOW": CX_COUNT_LOW,
    "CX_COUNT_HIGH": CX_COUNT_HIGH,
    "OPTIMIZATION_LEVEL": OPTIMIZATION_LEVEL,
    "BASIS_GATES": BASIS_GATES,
    "TRANSPILER_SEED": TRANSPILER_SEED,
    "CX_PER_SWAP": CX_PER_SWAP,
    "TEST_FRACTION": TEST_FRACTION,
    "package_versions": package_versions,
    "GRADIENT_BOOSTING_TREES": GRADIENT_BOOSTING_TREES,
    "GRADIENT_BOOSTING_DEPTH": GRADIENT_BOOSTING_DEPTH,
    "FEATURE_NAMES": FEATURE_NAMES,
}
dataset = {
    "config": config,
    "coupling_edges": coupling_edges,
    "circuits": circuits,                       # QuantumCircuit objects pickle fine
    "feature_rows": feature_rows,
    "swap_counts": swap_counts,
    "output_cx_counts": output_cx_counts,
    "physical_cx_estimate": physical_cx_estimate,
    "output_depths": output_depths,
    "input_depths": input_depths,
    "compile_seconds": compile_seconds,
    "train_indices": train_indices,
    "test_indices": test_indices,
}
with open(OUTPUT_PATH, "wb") as output_file:
    pickle.dump(dataset, output_file)
print("saved", OUTPUT_PATH)

In [ ]:
# Poster figure: predicted versus actual swap count on held-out circuits, both models.

figure, (left_axis, right_axis) = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
diagonal = [swap_counts.min(), swap_counts.max()]

left_axis.scatter(swap_counts[test_indices], cx_only_boosting_prediction, alpha=0.6)
left_axis.plot(diagonal, diagonal, "k--", linewidth=0.8)
left_axis.set_xlabel("actual swap count")
left_axis.set_ylabel("predicted swap count")
left_axis.set_title(f"Boosting on CX count only, MAE {cx_only_boosting_error.mean():.1f}")

right_axis.scatter(swap_counts[test_indices], all_features_prediction, alpha=0.6)
right_axis.plot(diagonal, diagonal, "k--", linewidth=0.8)
right_axis.set_xlabel("actual swap count")
right_axis.set_title(f"Boosting on interaction-graph features, MAE {all_features_error.mean():.1f}")

figure.suptitle(f"Routing cost on a {LATTICE_ROWS}x{LATTICE_COLUMNS} lattice, optimization level {OPTIMIZATION_LEVEL}, one compiler seed")
figure.tight_layout()
figure.savefig(FIGURE_PATH, dpi=200)
plt.show()
print("saved", FIGURE_PATH)

## Shred-outs

The base run predicts the cost of one compiler configuration on one hardware shape for one family of circuits, at one compiler seed. Each of those is a thing to vary. Pick one direction below, or invent your own. Each is written as the question your poster answers and the number that answers it. If your extension sweeps a setting, choose it on a validation split of circuits and report the test number once.

1. **Beat the compiler's own randomness.** SABRE routing is randomized. Compile each circuit with 20 different `seed_transpiler` values and record the best, worst, and mean swap count. How much of the base run's prediction error is just seed noise, and is best-of-20 a cheap win on its own? Metric: swap count spread across seeds, and the predictor's error against the mean instead of a single seed.

2. **Search the initial layout.** Instead of letting the compiler choose, place logical qubits on the lattice yourself. Try a breadth-first embedding of the interaction graph, a spectral ordering (Fiedler vector) snaked across the lattice, and random layouts, passing each through `transpile(..., initial_layout=...)`. Each candidate layout is scored by compiling it, so no learning is needed here. Does a structure-aware layout beat the compiler's default? Metric: swap count, your layouts versus the default, on the same circuits and the same seed.

3. **Compare routing passes.** Qiskit 2 ships three routing plugins you can name in `transpile(routing_method=...)`: `sabre`, `basic`, and `lookahead` (the old `stochastic` router was removed in Qiskit 2.0 and raises an error). To compare routers alone, fix the layout with `layout_method="trivial"` or an explicit `initial_layout`; otherwise you are comparing layout-plus-routing pipelines, which is also fine if you say so. Which router wins on which circuits, and can a classifier predict the winner from the interaction-graph features? Metric: swap count per router, and the classifier's accuracy at naming the best router.

4. **Dead qubits.** Simulate a device with defects by building a smaller coupling map explicitly: list the surviving physical qubits, relabel them 0 to k-1, and keep only the couplings between survivors. Deleting edges from the full map is not enough, because an isolated index is still a qubit the compiler may place onto. Make sure each circuit fits in the surviving connected component, or decide and record what happens when it does not. How does swap cost grow with the defect fraction, and does the predictor trained on the perfect lattice transfer? Metric: swap count versus defect fraction, and prediction error on the defective lattice.

5. **Structured circuits.** Random interaction graphs are unlike real algorithms. Build QAOA circuits on random graphs (see notebook hackaroo26-1-qaoa-warm-start), QFT circuits, and ladder circuits, and compile them. Does the predictor trained on random circuits transfer, and which structured family is cheapest to route per CX? Metric: prediction error per family, and swaps per CX per family.

6. **Predict physical cost, not swaps.** Hardware has no SWAP gate. Decompose each routed circuit into the basis without `swap` (transpile again with `BASIS_GATES` minus `swap`) and record the CX count and depth that come out; the pkl's `physical_cx_estimate` is the pre-optimization estimate. Change the regression target to physical CX count or to physical depth. Are the same features important? Metric: mean absolute error on the new target, and the importance ranking.

7. **A graph model instead of hand features.** Feed the interaction graph directly to a small graph neural network and predict the swap count. Note what the aggregate interaction graph throws away: gate order and multiplicity (two CX gates on the same pair are one edge). Add edge weights for multiplicity, or a sequence model that sees the gate list, and compare. Metric: mean absolute error on held-out random circuits and on the structured families from shred-out 5.

8. **Bigger lattices and bigger circuits.** Raise the lattice to 6x6 or 8x8 and the circuit sizes to match. Compile time grows and so does the swap count. Does the predictor's relative error hold? Metric: mean absolute error divided by mean swap count, versus lattice size.

Whatever you choose, load `compilation_cost_dataset.pkl` for the base-run circuits, features, and compile results. The `config` dict inside tells you exactly which run produced it, and `compile_seconds` are timings from the machine that ran it, not a portable quantity.

In [ ]:
# Your extension starts here.
